In [ ]:
from sfm.data.moving_librimix import MovingLibriMix as DS
from sfm.data.base_datamodule import BaseDatamodule as DM
from sfm.src.base_experiment import BaseExp as Exp
from sfm.data.utils import plot_acoustics
import yaml
from itertools import islice
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import torch
from IPython.display import Audio, display

N_SPEAKER = 2
EXAMPLE_IDX = 0
CONFIG_PATH = '../../config/'

In [ ]:
# load default parametrization
base_config_path = CONFIG_PATH + 'base_config.yaml'
with open(base_config_path, 'r') as file:
    config = yaml.safe_load(file)
config['audio_params']['n_speaker'] = N_SPEAKER
config['audio_params']['n_channel'] = 2

# instantiate dataloader
dm = DM(
    dataset=DS,
    batch_size=1, 
    n_workers=0, 
    **config,
)
val_dl = dm.val_dataloader()

# load example
example = next(islice(val_dl, EXAMPLE_IDX, None))

In [ ]:
# display waveform
fig, axs = plt.subplots(
    ncols=N_SPEAKER+1, nrows=2, figsize=(4 * (N_SPEAKER+1), 4), sharey=True, sharex=True
)
time = torch.arange(*example['audio_samples']) / config['audio_params']['sample_rate']
print('#'*30, f'example {EXAMPLE_IDX}', '#'*30)
axs[0, 0].set_title(f'mixture')
axs[0, 0].plot(time, example['mix_td'][0, config['exp_params']['ref_channel']])
axs[1, 0].set_xlabel('time in s')
for spk_idx in range(N_SPEAKER):
    axs[0, spk_idx+1].set_title(f'image speaker {spk_idx}')
    axs[0, spk_idx+1].plot(time, example['image_td'][0, spk_idx, config['exp_params']['ref_channel']])
    axs[1, spk_idx+1].set_title(f'dry speaker {spk_idx}')
    axs[1, spk_idx+1].plot(time, example['dry_td'][0, spk_idx, config['exp_params']['ref_channel']])
    axs[1, spk_idx+1].xaxis.set_major_locator(MultipleLocator(1))
    axs[1, spk_idx+1].set_xlabel('time in s')
plt.show()

# display audio
print('mixture')
display(Audio(example['mix_td'][0], rate=config['audio_params']['sample_rate']))
for spk_idx in range(N_SPEAKER):
    print(f'image speaker {spk_idx}')
    display(Audio(example['image_td'][0, spk_idx], rate=config['audio_params']['sample_rate']))
    print(f'dry speaker {spk_idx}')
    display(Audio(example['dry_td'][0, spk_idx], rate=config['audio_params']['sample_rate']))

In [ ]:
# plot acoustics
exp = Exp(model=None, **config)
plot_acoustics(exp, example, figsize=(10, 2))